# ExoData Challenge - Fase 1: Ingesta y Exploración
## Tarea 1.1: Cargar y auditar el dataset PSCompPars

In [ ]:
# CELDA 1 - Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / 'data'
DOCS_DIR = PROJECT_ROOT / 'docs'
DOCS_DIR.mkdir(exist_ok=True)

print(f"Python version: {sys.version}")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
# CELDA 2 - Leer CSV
dataset_path = DATA_DIR / 'PSCompPars_2026.csv'
df = pd.read_csv(dataset_path, comment='#', low_memory=False)

print(f"Archivo: {dataset_path}")
print(f"Shape: {df.shape}")
print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")

In [ ]:
# CELDA 3 - Inspección
print("=== PRIMERA FILA ===")
df.head(1).T

print("=== ÚLTIMA FILA ===")
df.tail(1).T

print("=== TIPOS DE DATOS ===")
df.dtypes.head(10)

In [ ]:
# CELDA 4 - Completitud por columna
completitud = df.notna().mean().sort_values(ascending=False)

reporte = pd.DataFrame({
    'completitud': completitud,
    'missing_pct': (1 - completitud) * 100
})

top_20 = reporte.head(20)
print("=== TOP 20 COLUMNAS MÁS COMPLETAS ===")
display(top_20)

top20_path = DOCS_DIR / 'top20_completitud.csv'
top_20.to_csv(top20_path, index=True)
print(f"\nReporte guardado en {top20_path}")

In [ ]:
# CELDA 5 - Variables CRÍTICAS
critical_vars = ['pl_rade', 'pl_bmasse', 'pl_orbper', 'pl_orbsmax', 'pl_eqt', 'st_teff', 'st_met', 'sy_pnum', 'discoverymethod', 'pl_insol']

print("=== COMPLETITUD DE VARIABLES CRÍTICAS ===")
for var in critical_vars:
    pct = df[var].notna().mean() * 100
    print(f"{var:<20}: {pct:>6.1f}%")

critical_vars_path = DOCS_DIR / 'critical_vars_completitud.csv'
df[critical_vars].notna().mean().to_csv(critical_vars_path)
print(f"\nReporte guardado en {critical_vars_path}")

In [ ]:
# CELDA 6 - Detectar outliers
for var in ['pl_orbper', 'pl_rade', 'pl_bmasse']:
    if var in df.columns:
        q99 = df[var].quantile(0.999)
        q01 = df[var].quantile(0.001)
        extreme_high = df[df[var] > q99].shape[0]
        extreme_low = df[df[var] < q01].shape[0]
        print(f"{var}: {extreme_high} valores > q99, {extreme_low} valores < q01")

In [ ]:
# CELDA 7 - Distribución por método de detección
method_counts = df['discoverymethod'].value_counts()

print("=== DISTRIBUCIÓN POR MÉTODO DE DETECCIÓN ===")
print(method_counts.head(10))
print(f"\nTotal de planetas con method conocido: {method_counts.sum()} de {len(df)}")

detection_method_path = DOCS_DIR / 'detection_method_distribution.csv'
method_counts.to_csv(detection_method_path)
print(f"\nReporte de sesgo guardado en {detection_method_path}")

In [ ]:
# CELDA 8 - Resumen ejecutivo
print("\n" + "="*60)
print("RESUMEN DE CALIDAD DE DATOS")
print("="*60)
print(f"✓ Total planetas: {len(df):,}")
print(f"✓ Total variables: {len(df.columns)}")
print(f"✓ Variables con >70% completitud: {(df.notna().mean() > 0.7).sum()}")
print(f"✓ Variables con >85% completitud: {(df.notna().mean() > 0.85).sum()}")
print("\n✓✓✓ DATASET CARGADO Y AUDITADO ✓✓✓")
print("\nSiguiente: Tarea 1.2 - Reporte de valores nulos detallado")